# Notebook 3: the TDI track

The neglected half of the challenge, and the one with the larger gap.

Everything we have ever submitted on TDI came from LightGBM over descriptors. **No
in-context tabular model has been run on this track at all**, and the entry at live rank 48
reports that TabICL specifically beats TabPFN here, which is the opposite of the
regression track. Their whole TDI entry is two models, a LightGBM baseline and TabICL on
frozen `chemprop_medium` embeddings, Caruana-combined, for an out-of-fold macro MCC of
0.283.

Where we stand, and what the gap looks like:

| | ours, out-of-fold AUC | a top entry |
| :--- | ---: | ---: |
| CYP2D6 | 0.5896, nested 0.6115 | 0.719 |
| CYP3A4 | 0.8186, nested 0.8113 | 0.864 |

CYP2D6 is barely above chance and is the largest documented gap anywhere in this project.
CYP3A4 is close enough that a decorrelated second arm might do it.

Labels: CYP2D6 has 1,495 with 324 positive, CYP3A4 has 2,346 with 764 positive. Both are
the small-data regime these models are built for.

AUC is the headline here, not MCC. AUC measures the ranker, which is what we are short of;
the calling threshold is already understood and is worth about +0.006.

**Run order.** LightGBM first as the control, then TabPFN, then TabICL. Those two conflict
on dependencies, so TabICL installs last and everything before it is already saved.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
# Challenge data including the is_TDI labels, folds, holdout mask, embeddings.
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
print(sorted(os.listdir('/content/cache')))

# Results go to Drive when it is available, so a runtime disconnect does not take
# them with it. Notebook 2 lost a 40-minute run that way.
import os
OUTDIR = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/cyp_results'
except Exception as e:
    print('Drive unavailable, falling back to the runtime disk:', type(e).__name__)
    print('If the runtime disconnects, this run is lost. Download early and often.')
os.makedirs(OUTDIR, exist_ok=True)
print('writing results to', OUTDIR)

In [ ]:
!pip -q install lightgbm

In [ ]:
import gc, json, os, time, numpy as np, pandas as pd, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from pathlib import Path
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import GroupKFold

CACHE = Path('/content/cache'); OUT = Path(OUTDIR)
ENDPOINTS = ['CYP2D6', 'CYP3A4']
N_FOLDS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

meta = pd.read_csv(CACHE / 'meta_train.csv.gz')
hold = np.load(CACHE / 'holdout_mask.npy')
G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=True)
EMB = np.load(CACHE / 'emb_medium_inhib.npy').astype(np.float32)
EMB_TE = np.load(CACHE / 'emb_medium_test.npy').astype(np.float32)
QH, QH_TE = np.load(CACHE / 'qhts_tr.npy'), np.load(CACHE / 'qhts_te.npy')
AUX, AUX_TE = np.load(CACHE / 'aux_train.npy'), np.load(CACHE / 'aux_test.npy')

X = np.nan_to_num(np.hstack([EMB, QH, AUX]).astype(np.float32))
X_TE = np.nan_to_num(np.hstack([EMB_TE, QH_TE, AUX_TE]).astype(np.float32))
BASELINE = {'CYP2D6': 0.5896, 'CYP3A4': 0.8186}   # what we have shipped, out-of-fold AUC

def labels(ep):
    y = meta[f'{ep}_is_TDI'].to_numpy(float)
    obs = np.isfinite(y)
    return y, np.flatnonzero(obs & ~hold), np.flatnonzero(obs & hold)

def best_mcc(y, p):
    """Highest MCC over all thresholds, read off the ROC rather than searched."""
    fpr, tpr, _ = roc_curve(y, p)
    pr = y.mean()
    tp, fn = tpr * pr, (1 - tpr) * pr
    fp, tn = fpr * (1 - pr), (1 - fpr) * (1 - pr)
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return float(np.nanmax(np.where(den > 0, (tp * tn - fp * fn) / np.maximum(den, 1e-12), 0)))

RESULTS = []
def run(name, fit_predict):
    for ep in ENDPOINTS:
        y, dev, ho = labels(ep)
        t0 = time.time()
        oof = np.zeros(len(dev))
        for tr, va in GroupKFold(N_FOLDS).split(dev, dev, G[dev]):
            oof[va] = fit_predict(X[dev[tr]], y[dev[tr]], X[dev[va]])
        hp = fit_predict(X[dev], y[dev], X[ho])
        allr = np.concatenate([dev, ho])
        tp = fit_predict(X[allr], y[allr], X_TE)
        np.savez_compressed(OUT / f'tdi_{name}_{ep}.npz',
                            oof=oof, dev=dev, hold=hp, ho=ho, test=tp)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        a, h = roc_auc_score(y[dev], oof), roc_auc_score(y[ho], hp)
        RESULTS.append(dict(arm=name, endpoint=ep, oof_auc=a, holdout_auc=h,
                            oof_best_mcc=best_mcc(y[dev], oof), baseline=BASELINE[ep],
                            seconds=round(time.time() - t0, 1)))
        json.dump(RESULTS, open(OUT / 'results.json', 'w'), indent=2)
        print(f'  {name:8s} {ep}: OOF AUC {a:.4f} (shipped {BASELINE[ep]:.4f}) | '
              f'holdout AUC {h:.4f} | best OOF MCC {RESULTS[-1]["oof_best_mcc"]:.4f} '
              f'({time.time()-t0:.0f}s)', flush=True)

for ep in ENDPOINTS:
    y, dev, ho = labels(ep)
    print(f'{ep}: {len(dev)} dev, {len(ho)} holdout, prevalence {y[dev].mean():.3f}')
print(f'features {X.shape[1]} columns, device {DEVICE}')

## Control: LightGBM, the family everything we shipped came from

Should land near 0.5896 and 0.8186. If it does not, the folds or features are not what the
earlier work used and the comparisons below are void.

In [ ]:
import lightgbm as lgb
P = dict(objective='binary', learning_rate=0.03, num_leaves=31, min_data_in_leaf=20,
         feature_fraction=0.3, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
         verbosity=-1, num_threads=4)
run('lgb', lambda a, b, c: lgb.train(P, lgb.Dataset(a, b), 700).predict(c))

In [ ]:
!pip -q install tabpfn==2.2.1

In [ ]:
from tabpfn import TabPFNClassifier
KW = dict(device=DEVICE, random_state=42, ignore_pretraining_limits=True)
run('tabpfn', lambda a, b, c: TabPFNClassifier(**KW).fit(a, b).predict_proba(c)[:, 1])

## TabICL

Installs last because it and TabPFN disagree on shared dependencies. If this cell forces a
runtime restart, re-run the payload cell and the setup cell, then come back; the LightGBM
and TabPFN arms are already written to disk.

In [ ]:
!pip -q install tabicl

In [ ]:
try:
    from tabicl import TabICLClassifier
    run('tabicl', lambda a, b, c: TabICLClassifier(device=DEVICE, random_state=42)
        .fit(a, b).predict_proba(c)[:, 1])
except Exception as e:
    print('TabICL unavailable in this runtime:', type(e).__name__, e)
    print('The LightGBM and TabPFN arms are saved and usable without it.')

In [ ]:
r = pd.DataFrame(RESULTS)
for col in ['oof_auc', 'holdout_auc', 'oof_best_mcc']:
    t = r.pivot(index='arm', columns='endpoint', values=col)
    t['MACRO'] = t.mean(axis=1)
    print(col); print(t.round(4).to_string()); print()
print('shipped out-of-fold AUC: CYP2D6 0.5896, CYP3A4 0.8186')
print('a top entry reports:      CYP2D6 0.719,  CYP3A4 0.864')

import shutil
from google.colab import files
shutil.make_archive('/content/notebook3_results', 'zip', OUT)
files.download('/content/notebook3_results.zip')